# 3. Data Preparation: Rekayasa 29 Fitur Spektral Fisik Terpadu dan Pembagian Dataset Seimbang

Tahap **Data Preparation** dalam metodologi **CRISP-DM** bertugas mentransformasikan data mentah hasil ekstraksi reflektansi Sentinel-2 menjadi representasi fitur bernilai tinggi yang siap digunakan oleh algoritma pembelajaran mesin. 

Pada tahap ini, dilakukan proses pembersihan data (*data cleaning*), rekayasa fitur biofisik lanjutan (*feature engineering*) menghasilkan **29 fitur prediktor (7 band reflektansi asli + 22 indeks biofisik spektral)**, pembagian sampel terstratifikasi (*Stratified 80:20 Train-Test Split*), serta penskalaan fitur tanpa menimbulkan kebocoran data (*data leakage*).

---

### Ruang Lingkup Aktivitas Data Preparation:
1. **Audit Integritas Data**: Memastikan dataset bebas dari nilai hilang (*missing value*), nilai tak terhingga (*infinity*), dan duplikasi baris, serta memverifikasi bahwa nilai reflektansi berada pada rentang fisik yang valid $[0.0, 1.0]$.
2. **Rekayasa 22 Indeks Biofisik Spektral**: Merumuskan indeks komparatif fisika optik untuk menangkap karakteristik kanopi vegetasi, badan air, substrat mangrove, dan kekedapan material bangunan.
3. **Pembagian Data Terstratifikasi (*Stratified 80:20 Split*)**: Membagi 250 data menjadi **200 sampel latih (40 sampel per kelas)** dan **50 sampel uji independen (10 sampel per kelas)** untuk menjamin evaluasi akhir yang obyektif dan seimbang.
4. **Penskalaan Fitur (*Feature Scaling*)**: Menerapkan `StandardScaler` yang dipelajari hanya dari data latih (*fit on train*) lalu diaplikasikan ke data uji (*transform on test*).


In [1]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('[OK] Seluruh library Data Preparation berhasil dimuat.')

[OK] Seluruh library Data Preparation berhasil dimuat.


## 3.1 Pemuatan Data dan Pemeriksaan Integritas Awal

Data mentah poligon dimuat dari direktori `2-data_understanding/extracted_data/sentinel2_sample_polygons.csv`. Sebelum dilakukan rekayasa fitur, dilakukan pemeriksaan sistematis terhadap:
* Jumlah baris dan kolom.
* Keberadaan nilai kosong (*missing/null values*).
* Keberadaan nilai numerik tak wajar (*infinite values*).
* Keberadaan sampel duplikat.
* Kesetaraan distribusi sampel antar kelas target.


In [2]:
data_dir = os.path.join('..', '2-data_understanding', 'extracted_data')
raw_path = os.path.join(data_dir, 'sentinel2_sample_polygons.csv')
df = pd.read_csv(raw_path)

print(f'Dimensi Dataset Awal: {df.shape[0]} baris, {df.shape[1]} kolom')
print(f'Total Nilai Hilang: {df.isnull().sum().sum()}')
print(f'Total Nilai Tak Terhingga: {np.isinf(df.select_dtypes(include=np.number)).values.sum()}')
print(f'Total Data Duplikat: {df.duplicated().sum()}')
print('\nDistribusi Sampel per Kelas:')
display(df['class_name'].value_counts())

Dimensi Dataset Awal: 250 baris, 18 kolom
Total Nilai Hilang: 0
Total Nilai Tak Terhingga: 0
Total Data Duplikat: 0

Distribusi Sampel per Kelas:


class_name
bangunan     50
hutan        50
mangrove     50
perairan     50
pertanian    50
Name: count, dtype: int64

## 3.2 Rekayasa 29 Fitur Spektral Fisik Terpadu (*Feature Engineering*)

Untuk membantu model membedakan kelas-kelas yang memiliki karakteristik spektral mirip (seperti Hutan vs Mangrove atau Bangunan vs Tanah Kering Pertanian), nilai pantulan 7 band asli diperkaya dengan **22 indeks biofisik turunan** yang dibagi ke dalam 3 kelompok utama:

### 1. Kelompok Indeks Vegetasi dan Aktivitas Klorofil:
* **NDVI (*Normalized Difference Vegetation Index*)**: $(B08 - B04) / (B08 + B04)$  
  Mengukur biomassa kanopi hijau aktif dan tingkat kehijauan vegetasi umum.
* **SAVI (*Soil Adjusted Vegetation Index*, $L=0.5$)**: $((B08 - B04) \times 1.5) / (B08 + B04 + 0.5)$  
  Menyesuaikan dan meredam pengaruh pantulan tanah latar belakang pada vegetasi berkepadatan rendah (misalnya lahan sawah awal tanam).
* **MSAVI (*Modified Soil Adjusted Vegetation Index*)**:  
  Formulasi koreksi tanah adaptif tanpa parameter konstan manual, mengisolasi kanopi muda dari pantulan tanah.
* **EVI (*Enhanced Vegetation Index*)**: $2.5 \times (B08 - B04) / (B08 + 6 \times B04 - 7.5 \times B02 + 1)$  
  Mengatasi kejenuhan (*saturation*) nilai NDVI pada tutupan kanopi hutan lebat dan meminimalkan gangguan hamburan atmosfer.
* **GNDVI (*Green Normalized Difference Vegetation Index*)**: $(B08 - B03) / (B08 + B03)$  
  Lebih sensitif terhadap konsentrasi klorofil daun pada kanopi lebat dibandingkan pita merah.
* **VR (*Vegetation Ratio*)**: $B08 / B04$  
  Rasio langsung pantulan inframerah dekat terhadap serapan merah untuk membedakan biomassa tinggi.

### 2. Kelompok Indeks Air, Kelembaban dan Substrat Mangrove:
* **MNDWI (*Modified Normalized Difference Water Index*)**: $(B03 - B11) / (B03 + B11)$  
  Membedakan badan air terbuka dari lahan terbangun dan tanah menggunakan pita SWIR-1.
* **AWEI_nsh (*Automated Water Extraction Index non-shadow*)**:  
  Mengekstraksi batas perairan terbuka secara stabil tanpa terganggu oleh pantulan permukaan tanah basah.
* **TC_Wetness (*Tasseled Cap Wetness*)**:  
  Transformasi linear multi-band untuk mengukur tingkat kelembaban fisik daun dan tanah.
* **B8A_B11_Ratio**: $B8A / B11$  
  Rasio Red-Edge terhadap SWIR-1 yang peka terhadap kadar air dalam jaringan daun tanaman pesisir.
* **Mangrove_Index_8A_11_4**: $(B8A - B04) / B11$  
  Diskriminator spesifik untuk mengisolasi kanopi mangrove dari hutan daratan dengan memanfaatkan penyerapan gelombang SWIR oleh lumpur rawa payau di bawah tajuk pohon.
* **NDMI_Narrow (*Narrow Normalized Difference Moisture Index*)**: $(B8A - B11) / (B8A + B11)$  
  Mengukur kadar air kanopi internal menggunakan pita sempit Red-Edge.
* **Mangrove_Substrate_Moisture**: $(B8A - B12) / (B8A + B12)$  
  Memanfaatkan pita SWIR-2 untuk mendeteksi substrat lumpur pasang-surut basah.
* **SWIR_Ratio**: $B11 / B12$  
  Rasio komparatif SWIR untuk membedakan tanah pertanian basah dari material buatan kering.

### 3. Kelompok Indeks Lahan Terbangun, Kekedapan dan Tanah:
* **NDBI (*Normalized Difference Built-up Index*)**: $(B11 - B08) / (B11 + B08)$  
  Mendeteksi kerapatan area terbangun berbasis tingginya reflektansi SWIR pada semen/genteng.
* **UI (*Urban Index*)**: $(B12 - B08) / (B12 + B08)$  
  Mengidentifikasi area perkotaan padat berbasis kontras SWIR-2 terhadap NIR.
* **EBBI (*Enhanced Built-up and Bareness Index*)**: $(B11 - B08) / (10 \times \sqrt{B11 + B08})$  
  Secara simultan memisahkan material bangunan dari tanah terbuka gersang.
* **BSI (*Bare Soil Index*)**: $((B11 + B04) - (B08 + B02)) / ((B11 + B04) + (B08 + B02))$  
  Mengidentifikasi tanah terbuka, lahan bera pertanian setelah panen, dan tegalan kering.
* **IBI (*Index-based Built-up Index*)**:  
  Menggabungkan NDBI, SAVI, dan MNDWI untuk meminimalisasi kesalahan klasifikasi lahan terbangun di perdesaan.
* **NDBI_NDVI_Diff**: $NDBI - NDVI$  
  Selisih kontras langsung antara karakter bangunan dan vegetasi.
* **Soil_Iron_Oxide**: $B04 / B02$  
  Mendeteksi kandungan mineral oksida besi pada tanah terbuka dan genteng tanah liat tradisional.
* **Urban_Impervious_Ratio**: $B11 / B02$  
  Mengukur tingkat kekedapan permukaan buatan manusia (*impervious surface*).

Kombinasi 7 band reflektansi dasar dan 22 indeks biofisik menghasilkan total **29 fitur prediktor** yang komprehensif.


In [3]:
eps = 1e-6
b02 = df['B02'].values
b03 = df['B03'].values
b04 = df['B04'].values
b08 = df['B08'].values
b8a = df['B8A'].values
b11 = df['B11'].values
b12 = df['B12'].values

# Indeks Spektral Fisik Terpadu
df['NDVI'] = (b08 - b04) / (b08 + b04 + eps)
df['MNDWI'] = (b03 - b11) / (b03 + b11 + eps)
df['NDBI'] = (b11 - b08) / (b11 + b08 + eps)
df['SAVI'] = ((b08 - b04) * 1.5) / (b08 + b04 + 0.5 + eps)
df['MSAVI'] = (2 * b08 + 1 - np.sqrt(np.maximum((2 * b08 + 1)**2 - 8 * (b08 - b04), 0))) / 2
df['Mangrove_Index_8A_11_4'] = (b8a - b04) / (b11 + eps)
df['BSI'] = ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02) + eps)
df['UI'] = (b12 - b08) / (b12 + b08 + eps)
df['EBBI'] = (b11 - b08) / (10.0 * np.sqrt(np.maximum(b11 + b08, 0.0) + eps))
df['NDBI_NDVI_Diff'] = df['NDBI'] - df['NDVI']

df['EVI'] = 2.5 * (b08 - b04) / (b08 + 6.0 * b04 - 7.5 * b02 + 1.0 + eps)
df['GNDVI'] = (b08 - b03) / (b08 + b03 + eps)
df['NDMI_Narrow'] = (b8a - b11) / (b8a + b11 + eps)
df['SWIR_Ratio'] = b11 / (b12 + eps)
df['Soil_Iron_Oxide'] = b04 / (b02 + eps)
df['AWEI_nsh'] = 4.0 * (b03 - b11) - (0.25 * b08 + 2.75 * b12)
df['TC_Wetness'] = 0.1511 * b02 + 0.1973 * b03 + 0.3283 * b04 + 0.3407 * b08 - 0.7117 * b11 - 0.4559 * b12
df['Mangrove_Substrate_Moisture'] = (b8a - b12) / (b8a + b12 + eps)
df['IBI'] = (df['NDBI'] - (df['SAVI'] + df['MNDWI'])/2.0) / (df['NDBI'] + (df['SAVI'] + df['MNDWI'])/2.0 + eps)
df['VR'] = b08 / (b04 + eps)
df['Urban_Impervious_Ratio'] = b11 / (b02 + eps)

class_map = {'bangunan': 0, 'pertanian': 1, 'perairan': 2, 'hutan': 3, 'mangrove': 4}
df['label'] = df['class_name'].map(class_map)

feature_cols = [c for c in df.columns if c not in ['sample_id', 'class_name', 'class_id', 'centroid_lon', 'centroid_lat', 'pixel_count', 'label']]
print(f'[OK] Berhasil merekayasa {len(feature_cols)} fitur spektral fisik.')
pd.set_option('display.max_columns', 35)
print(f'• 7 Band Spektral Dasar   : {feature_cols[:7]}')
print(f'• 22 Indeks Biofisik      : {feature_cols[7:]}')
print(f'• Total Fitur Prediktor   : {len(feature_cols)} Fitur')
display(df[['class_name'] + feature_cols].sample(5, random_state=42))

[OK] Berhasil merekayasa 29 fitur spektral fisik.
• 7 Band Spektral Dasar   : ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']
• 22 Indeks Biofisik      : ['NDVI', 'MNDWI', 'NDBI', 'B8A_B11_Ratio', 'Mangrove_Index_8A_11_4', 'SAVI', 'MSAVI', 'BSI', 'UI', 'EBBI', 'NDBI_NDVI_Diff', 'EVI', 'GNDVI', 'NDMI_Narrow', 'SWIR_Ratio', 'Soil_Iron_Oxide', 'AWEI_nsh', 'TC_Wetness', 'Mangrove_Substrate_Moisture', 'IBI', 'VR', 'Urban_Impervious_Ratio']
• Total Fitur Prediktor   : 29 Fitur


,class_name,B02,B03,B04,B08,B8A,B11,B12,NDVI,MNDWI,NDBI,B8A_B11_Ratio,Mangrove_Index_8A_11_4,SAVI,MSAVI,BSI,UI,EBBI,NDBI_NDVI_Diff,EVI,GNDVI,NDMI_Narrow,SWIR_Ratio,Soil_Iron_Oxide,AWEI_nsh,TC_Wetness,Mangrove_Substrate_Moisture,IBI,VR,Urban_Impervious_Ratio
142,mangrove,0.039469,0.067171,0.041383,0.324767,0.346018,0.141510,0.060059,0.773954,-0.356231,-0.393021,2.445184,2.152730,0.490765,0.487910,-0.331444,-0.687864,-0.026837,-1.166975,0.554764,0.657235,0.419479,2.356164,1.048471,-0.543709,0.015357,0.704199,1.412997,7.847650,3.585271
6,bangunan,0.121955,0.145679,0.172484,0.285956,0.294881,0.311119,0.239000,0.247518,-0.362170,0.042142,0.947809,0.393408,0.177589,0.160830,0.084902,-0.089447,0.003256,-0.205375,0.201736,0.324988,-0.026795,1.301743,1.414314,-1.390497,-0.129162,0.104668,-2.680758,1.657862,2.551072
97,hutan,0.020943,0.043737,0.024382,0.207982,0.222597,0.127510,0.064487,0.790133,-0.489188,-0.239862,1.745727,1.554496,0.376040,0.341867,-0.202282,-0.526645,-0.013893,-1.029995,0.383391,0.652489,0.271595,1.977267,1.164171,-0.564424,-0.029490,0.550744,0.618307,8.529655,6.088142
60,hutan,0.093009,0.111531,0.095273,0.341891,0.364254,0.213985,0.148287,0.564129,-0.314743,-0.230098,1.702241,1.257002,0.394729,0.377623,-0.168837,-0.394967,-0.017155,-0.794227,0.507041,0.508048,0.259873,1.443040,1.024334,-0.903077,-0.036078,0.421366,1.420754,3.588491,2.300665
112,mangrove,0.039188,0.070652,0.036872,0.421666,0.436047,0.141268,0.052005,0.839172,-0.333220,-0.498100,3.086671,2.825641,0.602156,0.639033,-0.442434,-0.780416,-0.037372,-1.337272,0.713111,0.712982,0.510603,2.716387,0.940899,-0.530894,0.051378,0.786887,1.739584,11.435477,3.604823


## 3.3 Pembagian Data (Stratified 80:20 Split) dan Penskalaan Fitur

Untuk memastikan model dievaluasi secara obyektif pada data yang belum pernah dipelajari, dataset dibagi dengan rasio 80:20:
* **Data Latih (*Train Set*)**: 200 sampel (40 sampel per kelas seimbang sempurna).
* **Data Uji (*Test Set*)**: 50 sampel (10 sampel per kelas seimbang sempurna).

### Pencegahan Kebocoran Data (*Data Leakage Prevention*):
Penskalaan fitur dilakukan menggunakan `StandardScaler`. Parameter rata-rata ($\mu$) dan simpangan baku ($\sigma$) dihitung (*fit*) hanya dari data latih, kemudian nilai parameter tersebut diterapkan (*transform*) pada data latih dan data uji secara terpisah. Hal ini menjamin bahwa informasi distribusi data uji tidak bocor ke dalam proses pelatihan model.


In [4]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=2024, stratify=df['label'])

prep_dir = 'prepared_data'
os.makedirs(prep_dir, exist_ok=True)

df.to_csv(os.path.join(prep_dir, 'data_prepared_all.csv'), index=False)
train_df.to_csv(os.path.join(prep_dir, 'train_data.csv'), index=False)
test_df.to_csv(os.path.join(prep_dir, 'test_data.csv'), index=False)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(train_df[feature_cols])
X_te_scaled = scaler.transform(test_df[feature_cols])

train_scaled_df = pd.DataFrame(X_tr_scaled, columns=feature_cols)
train_scaled_df['label'] = train_df['label'].values
train_scaled_df.to_csv(os.path.join(prep_dir, 'train_scaled.csv'), index=False)

test_scaled_df = pd.DataFrame(X_te_scaled, columns=feature_cols)
test_scaled_df['label'] = test_df['label'].values
test_scaled_df.to_csv(os.path.join(prep_dir, 'test_scaled.csv'), index=False)

joblib.dump(scaler, os.path.join(prep_dir, 'standard_scaler.pkl'))

print(f'Data Latih (Train Set) : {train_df.shape[0]} sampel (40 per kelas)')
print(f'Data Uji (Test Set)    : {test_df.shape[0]} sampel (10 per kelas)')
print('[OK] Seluruh dataset bersih berhasil disimpan ke prepared_data/.')

Data Latih (Train Set) : 200 sampel (40 per kelas)
Data Uji (Test Set)    : 50 sampel (10 per kelas)
[OK] Seluruh dataset bersih berhasil disimpan ke prepared_data/.
